<a href="https://colab.research.google.com/github/pepper-0/java-cookies/blob/training-data/java_cookies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
from pathlib import Path

# For local use, load KAGGLE_USERNAME and KAGGLE_KEY from the repository's .env.
# Install the project requirements first so python-dotenv is available.
try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd() / ".env")
except ImportError:
    pass

# Colab runtimes cannot read your computer's local .env file. Fetch credentials
# from Colab Secrets instead (create secrets named KAGGLE_USERNAME and KAGGLE_KEY).
try:
    from google.colab import userdata
    for secret_name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        if not os.getenv(secret_name):
            try:
                secret_value = userdata.get(secret_name)
                if secret_value:
                    os.environ[secret_name] = secret_value
            except Exception:
                pass
except ImportError:
    pass

kaggle_username = os.getenv("KAGGLE_USERNAME")
kaggle_key = os.getenv("KAGGLE_KEY")
if not kaggle_username or not kaggle_key:
    raise RuntimeError(
        "Kaggle credentials are missing. Set KAGGLE_USERNAME and KAGGLE_KEY "
        "in your local .env or Google Colab Secrets."
    )

# Kaggle CLI expects this legacy username/key JSON format.
kaggle_creds = {"username": kaggle_username, "key": kaggle_key}
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
with (kaggle_dir / "kaggle.json").open("w", encoding="utf-8") as credentials_file:
    json.dump(kaggle_creds, credentials_file)
os.chmod(kaggle_dir / "kaggle.json", 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)

print("Kaggle credentials configured (values are not displayed).")

Kaggle credentials configured (values are not displayed).


In [2]:
import os
import shutil
import zipfile

# Keep the Kaggle config directory chosen in the credential setup cell.
os.environ.setdefault('KAGGLE_CONFIG_DIR', os.path.join(os.path.expanduser('~'), '.kaggle'))

# ==========================================
# 1. SETUP DIRECTORIES
# ==========================================
RAW_DIR = "/content/raw_datasets"
PROCESSED_DIR = "/content/processed_dataset"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ==========================================
# 2. DOWNLOAD ALL DATASETS (RELIABLE & VERBOSE)
# ==========================================
print("--- 1. DOWNLOADING KAGGLE DATASETS ---")
# Download without --unzip to prevent CLI argument incompatibility issues
!kaggle datasets download -d visalakshiiyer/cassava-image-dataset -p {RAW_DIR}/cassava --force
!kaggle datasets download -d kaustavbiswal/maize-diseases -p {RAW_DIR}/maize --force
!kaggle datasets download -d manhhoangvan/fieldplant -p {RAW_DIR}/fieldplant --force

print("\n--- 2. CLONING GITHUB REPOSITORIES ---")
# Clone cleanly by removing existing folders first if present
for repo, url in [
    ("rice", "https://github.com/ai-agriculture-circuits-and-systems/rice_leaf_disease_classification.git"),
    ("plantvillage", "https://github.com/spMohanty/PlantVillage-Dataset.git"),
    ("plantdoc", "https://github.com/pratikkayal/PlantDoc-Dataset.git")
]:
    repo_path = os.path.join(RAW_DIR, repo)
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {url} {repo_path}

print("\n--- 3. DOWNLOADING MENDELEY DATA ---")
# Using curl with progress bar to guarantee it doesn't freeze silently
!curl -L -o {RAW_DIR}/mendeley_crop.zip "https://data.mendeley.com/public-api/zip/bwh3zbpkpv/download/1"

print("\nAll downloads completed successfully!")

# ==========================================
# 3. UNPACK & IMMEDIATELY DELETE ZIP ARCHIVES
# ==========================================
print("\n--- 4. EXTRACTING & IMMEDIATELY DELETING ARCHIVES ---")
for root, dirs, files in os.walk(RAW_DIR):
    for file in files:
        if file.endswith('.zip'):
            zip_path = os.path.join(root, file)
            extract_to = os.path.join(root, os.path.splitext(file)[0])
            os.makedirs(extract_to, exist_ok=True)
            try:
                print(f"Extracting {file} to {extract_to}...")
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(extract_to)
                # Delete the ZIP immediately after successful extraction to save space
                os.remove(zip_path)
                print(f"Deleted source archive {file} to free space.")
            except Exception as e:
                print(f"Could not process {file}: {e}")

--- 1. DOWNLOADING KAGGLE DATASETS ---
Dataset URL: https://www.kaggle.com/datasets/visalakshiiyer/cassava-image-dataset
License(s): CC0-1.0
100% 842M/842M [00:06<00:00, 132MB/s] 

Dataset URL: https://www.kaggle.com/datasets/kaustavbiswal/maize-diseases
License(s): CC-BY-NC-SA-4.0
100% 8.10G/8.10G [01:15<00:00, 115MB/s] 

Dataset URL: https://www.kaggle.com/datasets/manhhoangvan/fieldplant
License(s): unknown
100% 5.51G/5.51G [00:28<00:00, 207MB/s]


--- 2. CLONING GITHUB REPOSITORIES ---
Cloning into '/content/raw_datasets/rice'...
remote: Enumerating objects: 7788, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7788 (delta 0), reused 1 (delta 0), pack-reused 7785 (from 3)
Receiving objects: 100% (7788/7788), 986.43 MiB | 2.31 MiB/s, done.
Resolving deltas: 100% (3833/3833), done.
Updating files: 100% (15380/15380), done.
Cloning into '/content/raw_datasets/plantvillage'...
remote: Enumerating objects: 163264, done.
remo

In [3]:
import os
import shutil

print("--- 1. MONITORING SPACE BEFORE CLEANUP ---")
!df -h /content

# ==========================================
# DELETE ALL ZIP ARCHIVES TO FREE SPACE
# ==========================================
print("\n--- 2. DELETING HEAVY ZIP ARCHIVES ---")
zips_deleted = 0
for root, dirs, files in os.walk(RAW_DIR):
    for file in files:
        if file.endswith('.zip'):
            zip_path = os.path.join(root, file)
            size_mb = os.path.getsize(zip_path) / (1024 * 1024)
            try:
                os.remove(zip_path)
                zips_deleted += 1
                print(f"Deleted {file} (~{size_mb:.1f} MB)")
            except Exception as e:
                print(f"Failed to delete {file}: {e}")

# ==========================================
# REMOVE LARGE GIT HISTORY FILES (.git Folders)
# ==========================================
print("\n--- 3. REMOVING UNNECESSARY .git DIRECTORIES ---")
for root, dirs, files in os.walk(RAW_DIR):
    for d in list(dirs):
        if d == ".git":
            git_path = os.path.join(root, d)
            try:
                shutil.rmtree(git_path)
                print(f"Cleaned git history directory: {git_path}")
            except Exception as e:
                print(f"Failed to delete {git_path}: {e}")

print(f"\nDeleted {zips_deleted} large archive files.")
print("--- 4. SPACE AVAILABLE AFTER CLEANUP ---")
!df -h /content

--- 1. MONITORING SPACE BEFORE CLEANUP ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   75G   38G  67% /

--- 2. DELETING HEAVY ZIP ARCHIVES ---

--- 3. REMOVING UNNECESSARY .git DIRECTORIES ---
Cleaned git history directory: /content/raw_datasets/plantdoc/.git
Cleaned git history directory: /content/raw_datasets/rice/.git
Cleaned git history directory: /content/raw_datasets/plantvillage/.git

Deleted 0 large archive files.
--- 4. SPACE AVAILABLE AFTER CLEANUP ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   71G   42G  63% /


In [4]:
import os
import shutil
import random
from PIL import Image

# ==============================================================================
# 1. CONFIGURATION & TARGET TAXONOMY SETUP
# ==============================================================================
RAW_DIR = "/content/raw_datasets"
PROCESSED_DIR = "/content/processed_dataset"
MIN_IMAGE_SIZE = 150
MAX_IMAGES_PER_CLASS = 1500

TARGET_CLASSES = [
    "maize_healthy", "maize_streak_disease", "maize_northern_leaf_blight", "maize_common_rust",
    "cassava_healthy", "cassava_brown_streak_disease", "cassava_bacterial_blight", "cassava_mosaic_disease",
    "rice_healthy", "rice_bacterial_blight", "rice_blast", "rice_yellow_mottle_disease"
]

# ==============================================================================
# 2. SPACE-OPTIMIZED PARSING, CLEANING, & MOVING SCRIPT (DESTRUCTIVE PIPELINE)
# ==============================================================================
def clean_move_and_prune_images(source_root, target_root):
    print("Starting ultra space-optimized dataset cleaning and moving with EXPANDED mapping...")
    print("Moving matched images and deleting unused raw images in-place to save disk space.")

    # Robust, comprehensive mapping rules matching folder structures of all downloaded sources
    keyword_mapping = {
        "maize_healthy": [
            "maize_healthy", "corn_healthy", "healthy_maize", "healthy maize", "corn healthy",
            "corn___healthy", "maize___healthy", "healthy/corn", "healthy/maize", "maize/healthy", "corn/healthy", "/maize/", "healthy"
        ],
        "maize_streak_disease": [
            "streak", "msv", "maize_streak", "maize streak", "maize_streak_disease"
        ],
        "maize_northern_leaf_blight": [
            "blight", "northern_leaf_blight", "nlb", "corn_blight", "northern leaf blight",
            "corn___northern_leaf_blight", "corn___northern leaf blight", "corn_northern_leaf_blight", "nlb", "corn_blight"
        ],
        "maize_common_rust": [
            "rust", "common_rust", "maize_rust", "common rust", "maize common rust",
            "corn___common_rust", "corn___common rust", "corn_common_rust", "maize_rust"
        ],

        "cassava_healthy": [
            "healthy1", "cassava_healthy", "healthy_cassava", "healthy cassava", "cassava___healthy",
            "class 4", "/4/", "healthy/cassava", "cassava_healthy_leaf"
        ],
        "cassava_brown_streak_disease": [
            "brown_streak", "cbsd", "brown streak", "cassava_brown_streak",
            "class 1", "/1/", "brown_streak_disease"
        ],
        "cassava_bacterial_blight": [
            "bacterial_blight", "cbb", "bacterial blight", "cassava_bacterial_blight",
            "class 0", "/0/", "bacterial_blight_disease"
        ],
        "cassava_mosaic_disease": [
            "mosaic", "cmd", "cassava_mosaic", "cassava mosaic",
            "class 3", "/3/", "mosaic_disease"
        ],

        "rice_healthy": [
            "rice_healthy", "healthy_rice", "healthy rice", "rice___healthy", "healthy/rice"
        ],
        "rice_bacterial_blight": [
            "bacterial_blight", "bacterial blight", "bb", "rice_bacterial_blight", "rice bacterial blight"
        ],
        "rice_blast": [
            "blast", "rice_blast", "magnaporthe", "rice blast"
        ],
        "rice_yellow_mottle_disease": [
            "yellow_mottle", "rym", "rymvd", "mottle", "yellow mottle", "rice_yellow_mottle"
        ]
    }

    # Keep an in-memory index count to avoid slow os.listdir inside loop
    class_counters = {}
    for target_cls in TARGET_CLASSES:
        target_class_dir = os.path.join(target_root, target_cls)
        if os.path.exists(target_class_dir):
            class_counters[target_cls] = len(os.listdir(target_class_dir))
        else:
            class_counters[target_cls] = 0

    moved_count = 0
    deleted_unmatched_count = 0
    skipped_corrupted_count = 0
    skipped_small_count = 0

    # Walk top-down to allow directory deletions
    for root, dirs, files in os.walk(source_root, topdown=False):
        parent_folder = os.path.basename(root).lower()
        grandparent_folder = os.path.basename(os.path.dirname(root)).lower()
        combined_path_context = f"{grandparent_folder}/{parent_folder}"

        for file in files:
            file_path = os.path.join(root, file)

            # 1. Instantly delete non-image configuration files and system files
            if not file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                try:
                    os.remove(file_path)
                except Exception:
                    pass
                continue

            filename_lower = file.lower()

            # 2. Check keyword matching (Fast string parsing)
            # Create a combined string of the full path for context
            combined_path_context = f"{root.lower()}/{file.lower()}"
            filename_lower = file.lower()

            matched_target_class = None

            # 1. RICE LOGIC: Use your original, working loose-matching logic directly
            if "rice" in combined_path_context:
                for target_cls, keywords in keyword_mapping.items():
                    if target_cls.startswith("rice_"):
                        if any(kw in combined_path_context or kw in filename_lower for kw in keywords):
                            matched_target_class = target_cls
                            break

            # 2. MAIZE & CASSAVA LOGIC: Use the strict path-isolated filtering
            else:
                # Determine crop type for non-rice files
                if "maize" in combined_path_context or "corn" in combined_path_context or "kaustavbiswal" in combined_path_context:
                    file_crop = "maize"
                elif "cassava" in combined_path_context:
                    file_crop = "cassava"
                else:
                    file_crop = "unknown"

                if file_crop != "unknown":
                    for target_cls, keywords in keyword_mapping.items():
                        if target_cls.startswith(file_crop):
                            if any(kw in combined_path_context for kw in keywords):
                                matched_target_class = target_cls
                                break

            # If it doesn't match our desired targets, delete it immediately
            if not matched_target_class:
                try:
                    os.remove(file_path)
                    deleted_unmatched_count += 1
                except Exception:
                    pass
                continue

            # 3. Fast validation of size and integrity
            try:
                with Image.open(file_path) as img:
                    width, height = img.size
                    if width < MIN_IMAGE_SIZE or height < MIN_IMAGE_SIZE:
                        skipped_small_count += 1
                        os.remove(file_path)  # Delete small image
                        continue
                    img.draft(img.mode, (32, 32))  # Minimal loading decoding test

                target_class_dir = os.path.join(target_root, matched_target_class)
                os.makedirs(target_class_dir, exist_ok=True)

                existing_count = class_counters[matched_target_class]
                ext = os.path.splitext(file)[1].lower()
                dest_path = os.path.join(target_class_dir, f"img_{existing_count:05d}{ext}")

                # Move instead of copy to free up space instantly
                shutil.move(file_path, dest_path)
                class_counters[matched_target_class] += 1
                moved_count += 1

            except (IOError, SyntaxError, Image.DecompressionBombError):
                skipped_corrupted_count += 1
                try:
                    os.remove(file_path)  # Delete corrupted file
                except Exception:
                    pass
                continue

        # 4. Clean up empty subdirectories to free directory entries
        if not os.listdir(root) and root != source_root:
            try:
                os.rmdir(root)
            except Exception:
                pass

    print("\n--- Cleanup & Processing Complete ---")
    print(f"Successfully parsed and MOVED to processed_dataset: {moved_count} images.")
    print(f"Deleted unmatched images: {deleted_unmatched_count}")
    print(f"Deleted too-small images: {skipped_small_count}")
    print(f"Deleted corrupted images: {skipped_corrupted_count}")

# ==============================================================================
# 3. DATASET AUDITING, CLASS BALANCING, & CROP-TYPE GROUPING SCRIPT
# ==============================================================================
def audit_and_balance_dataset(processed_root, max_images_per_class=1500):
    print("\n--- Auditing Processed Dataset & Crop Hierarchies ---")

    crop_groups = {
        "Maize": ["maize_healthy", "maize_streak_disease", "maize_northern_leaf_blight", "maize_common_rust"],
        "Cassava": ["cassava_healthy", "cassava_brown_streak_disease", "cassava_bacterial_blight", "cassava_mosaic_disease"],
        "Rice": ["rice_healthy", "rice_bacterial_blight", "rice_blast", "rice_yellow_mottle_disease"]
    }

    total_dataset_size = 0

    for crop_name, classes in crop_groups.items():
        print(f"\n[Crop Family: {crop_name}]")
        crop_total = 0

        for class_name in classes:
            class_dir = os.path.join(processed_root, class_name)
            if os.path.exists(class_dir):
                images = os.listdir(class_dir)
                count = len(images)

                if count > max_images_per_class:
                    print(f"  -> Downsampling {class_name} from {count} down to {max_images_per_class}...")
                    excess_images = random.sample(images, count - max_images_per_class)
                    for img in excess_images:
                        os.remove(os.path.join(class_dir, img))
                    count = max_images_per_class

                crop_total += count
                total_dataset_size += count
                print(f"   - {class_name}: {count} images ready")
            else:
                print(f"   - {class_name}: 0 images found (Folder missing)")

        print(f"Total images for {crop_name}: {crop_total}")

    print(f"\nOverall Processed Dataset Size: {total_dataset_size} images ready for MobileNetV2 training.")

# Execute cleaning, mapping, and balancing automatically
clean_move_and_prune_images(RAW_DIR, PROCESSED_DIR)
audit_and_balance_dataset(PROCESSED_DIR, max_images_per_class=MAX_IMAGES_PER_CLASS)

Starting ultra space-optimized dataset cleaning and moving with EXPANDED mapping...
Moving matched images and deleting unused raw images in-place to save disk space.

--- Cleanup & Processing Complete ---
Successfully parsed and MOVED to processed_dataset: 78859 images.
Deleted unmatched images: 265875
Deleted too-small images: 6
Deleted corrupted images: 39

--- Auditing Processed Dataset & Crop Hierarchies ---

[Crop Family: Maize]
  -> Downsampling maize_healthy from 49514 down to 1500...
   - maize_healthy: 1500 images ready
   - maize_streak_disease: 13 images ready
  -> Downsampling maize_northern_leaf_blight from 3139 down to 1500...
   - maize_northern_leaf_blight: 1500 images ready
  -> Downsampling maize_common_rust from 3692 down to 1500...
   - maize_common_rust: 1500 images ready
Total images for Maize: 4513

[Crop Family: Cassava]
   - cassava_healthy: 641 images ready
   - cassava_brown_streak_disease: 50 images ready
  -> Downsampling cassava_bacterial_blight from 14435

### 1. PyTorch MobileNetV3 Training Pipeline
The following code sets up data loaders from `/content/processed_dataset`, partitions them into an 80/20 train/validation split, loads a pretrained **MobileNetV3-Large** architecture, adapts the classification head to output predictions for the classes found in the processed dataset, and starts the training epochs.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import os
from PIL import ImageFile

# Enable loading of truncated/broken images to prevent DataLoader crashes
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 1. Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Paths & Hyperparameters
DATA_DIR = "/content/processed_dataset"
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

# 3. Data Transformations (Optimized for MobileNetV3 input resolution 224x224)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. Load Dataset and Partition (80% Train, 20% Val)
full_dataset = datasets.ImageFolder(root=DATA_DIR)
num_classes = len(full_dataset.classes)
print(f"Detected {num_classes} classes: {full_dataset.classes}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# Apply subset-specific transforms
train_subset.dataset.transform = train_transform
val_subset.dataset.transform = val_transform

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Data Split completed: Train Size = {len(train_subset)}, Val Size = {len(val_subset)}")

Using device: cuda
Detected 10 classes: ['cassava_bacterial_blight', 'cassava_brown_streak_disease', 'cassava_healthy', 'cassava_mosaic_disease', 'maize_common_rust', 'maize_healthy', 'maize_northern_leaf_blight', 'maize_streak_disease', 'rice_blast', 'rice_healthy']
Data Split completed: Train Size = 8621, Val Size = 2156


In [6]:
# 5. Initialize Pretrained MobileNetV3-Large Model
model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)

# Replace classification head to match 12 classes
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, num_classes)
model = model.to(device)

# 6. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("MobileNetV3 Model ready for training!")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 146MB/s]


MobileNetV3 Model ready for training!


In [7]:
# 7. Training Loop
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(train_subset)
    epoch_acc = 100. * correct / total

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_epoch_loss = val_loss / len(val_subset)
    val_epoch_acc = 100. * val_correct / val_total

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}% | Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.2f}%")

# Save the final weights
torch.save(model.state_dict(), "mobilenetv3_crop_diseases.pth")
print("Training Complete! Saved weights as mobilenetv3_crop_diseases.pth")

Epoch [1/10] | Train Loss: 0.3083 Acc: 91.01% | Val Loss: 0.3408 Acc: 88.78%
Epoch [2/10] | Train Loss: 0.1564 Acc: 95.17% | Val Loss: 0.2309 Acc: 93.00%
Epoch [3/10] | Train Loss: 0.1242 Acc: 96.24% | Val Loss: 0.1256 Acc: 96.61%
Epoch [4/10] | Train Loss: 0.1089 Acc: 96.51% | Val Loss: 0.1868 Acc: 94.67%
Epoch [5/10] | Train Loss: 0.1025 Acc: 96.91% | Val Loss: 0.1940 Acc: 94.57%
Epoch [6/10] | Train Loss: 0.0743 Acc: 97.75% | Val Loss: 0.1149 Acc: 96.47%
Epoch [7/10] | Train Loss: 0.0743 Acc: 97.81% | Val Loss: 0.1729 Acc: 95.83%
Epoch [8/10] | Train Loss: 0.1129 Acc: 96.66% | Val Loss: 0.2416 Acc: 92.76%
Epoch [9/10] | Train Loss: 0.0684 Acc: 97.97% | Val Loss: 0.1324 Acc: 96.10%
Epoch [10/10] | Train Loss: 0.0608 Acc: 98.16% | Val Loss: 0.2798 Acc: 94.57%
Training Complete! Saved weights as mobilenetv3_crop_diseases.pth


### 2. Export and Download Model Weights
Below, we will save both the weights configuration (`state_dict`) and the entire model, then download the lightweight `.pth` file to your local computer.

In [8]:
import torch
from google.colab import files

# --- Method 1: Save Model Weights (Highly Recommended) ---
# This is already saved at the end of your training loop as 'mobilenetv3_crop_diseases.pth'.
# Let's ensure it is saved safely to disk:
weights_filename = "mobilenetv3_crop_diseases.pth"
torch.save(model.state_dict(), weights_filename)
print(f"✓ Model weights saved to: {weights_filename}")

# --- Method 2: Save the Full Model (Architecture + Weights) ---
full_model_filename = "mobilenetv3_crop_diseases_full.pth"
torch.save(model, full_model_filename)
print(f"✓ Full model saved to: {full_model_filename}")

# --- Method 3: Download the weights file directly to your local machine ---
try:
    print("Triggering browser download... (Check your downloads folder shortly)")
    files.download(weights_filename)
except Exception as e:
    print(f"Could not automatically download the file: {e}")
    print("Alternative: You can manually download 'mobilenetv3_crop_diseases.pth' from the left folder sidebar.")

✓ Model weights saved to: mobilenetv3_crop_diseases.pth
✓ Full model saved to: mobilenetv3_crop_diseases_full.pth
Triggering browser download... (Check your downloads folder shortly)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 3. Backup Model Weights to Google Drive (Persistent Storage)
Run the following cell to mount your Google Drive and save your trained MobileNetV3 weights safely so they won't get deleted when Colab refreshes.

In [9]:
from google.colab import drive
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Copy weights to your Drive
source_path = "mobilenetv3_crop_diseases.pth"
destination_dir = "/content/drive/MyDrive/crop_disease_model/"

import os
os.makedirs(destination_dir, exist_ok=True)
shutil.copy(source_path, os.path.join(destination_dir, source_path))

print(f"✓ Successfully saved a backup of your model weights to Google Drive: {destination_dir + source_path}")

MessageError: Error: credential propagation was unsuccessful

### 4. How to Use the Model for Inference (Application Backend Integration)

To use your trained model in an application (like a web or mobile app backend), you need to perform the following steps for every uploaded image:
1. **Load the Model Architecture** (MobileNetV3) and adapted classification head.
2. **Load the Trained Weights** (`mobilenetv3_crop_diseases.pth`).
3. **Preprocess the Uploaded Image** (Resize to 224x224, convert to a Tensor, and apply ImageNet normalization).
4. **Run Inference** (Forward pass through the model under `torch.no_grad()`).
5. **Map Output Index** to the actual human-readable class name.

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# Keep this order identical to full_dataset.classes from the training run.
CLASS_NAMES = [
    "cassava_bacterial_blight",
    "cassava_mosaic_disease",
    "maize_common_rust",
    "maize_northern_leaf_blight",
    "maize_streak_disease",
    "rice_bacterial_blight",
    "rice_blast",
    "rice_healthy",
    "rice_yellow_mottle_disease",
]


def load_crop_model(weights_path, num_classes=len(CLASS_NAMES), device="cpu"):
    if num_classes != len(CLASS_NAMES):
        raise ValueError(f"This checkpoint expects {len(CLASS_NAMES)} classes.")

    model = models.mobilenet_v3_large(weights=None)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    state_dict = torch.load(weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def predict_crop_disease(image_path, model, device="cpu"):
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    with Image.open(image_path) as image:
        input_tensor = preprocess(image.convert("RGB")).unsqueeze(0).to(device)

    with torch.inference_mode():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
        confidence, class_idx = torch.max(probabilities, dim=0)

    return CLASS_NAMES[class_idx.item()], confidence.item()


# Demonstrate inference on a sample image, if the training data is available.
test_class = "rice_healthy"
sample_dir = f"/content/processed_dataset/{test_class}"
if os.path.exists(sample_dir) and os.listdir(sample_dir):
    sample_img_path = os.path.join(sample_dir, os.listdir(sample_dir)[0])
    inference_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    crop_model = load_crop_model(
        "mobilenetv3_crop_diseases.pth",
        device=inference_device,
    )
    label, score = predict_crop_disease(
        sample_img_path,
        crop_model,
        device=inference_device,
    )
    print(f"Target True Class: {test_class}")
    print(f"Predicted Class:   {label}")
    print(f"Confidence Score:  {score * 100:.2f}%")
else:
    print("No sample images found to demonstrate inference.")

Target True Class: rice_healthy
Predicted Class:   rice_healthy
Confidence Score:  99.99%


In [ ]:
import os

print("--- VERIFYING MODEL FILES ON DISK ---")
expected_files = [
    "/content/mobilenetv3_crop_diseases.pth",
    "/content/mobilenetv3_crop_diseases_full.pth",
    "/content/drive/MyDrive/crop_disease_model/mobilenetv3_crop_diseases.pth"
]

for path in expected_files:
    exists = os.path.exists(path)
    status = "✅ FOUND" if exists else "❌ MISSING"
    size_mb = f"({os.path.getsize(path) / (1024*1024):.2f} MB)" if exists else ""
    print(f"{status:10} | {path} {size_mb}")


In [ ]:
import os

print("=== LOCATING YOUR TRAINED MODELS ===\n")

# 1. Local Workspace Locations
local_weights = "/content/mobilenetv3_crop_diseases.pth"
local_full_model = "/content/mobilenetv3_crop_diseases_full.pth"

# 2. Google Drive Persistent Backup Location
drive_weights = "/content/drive/MyDrive/crop_disease_model/mobilenetv3_crop_diseases.pth"

# Display Status
print("📍 [Local Workspace Directory - /content/]")
if os.path.exists(local_weights):
    print(f"  ✅ Model Weights saved to: {local_weights} ({os.path.getsize(local_weights) / (1024*1024):.2f} MB)")
else:
    print(f"  ❌ Model Weights missing at: {local_weights}")

if os.path.exists(local_full_model):
    print(f"  ✅ Full Model saved to:      {local_full_model} ({os.path.getsize(local_full_model) / (1024*1024):.2f} MB)")
else:
    print(f"  ❌ Full Model missing at:      {local_full_model}")

print("\n📍 [Google Drive Backup Directory]")
if os.path.exists(drive_weights):
    print(f"  ✅ Persistent Backup safe at: {drive_weights} ({os.path.getsize(drive_weights) / (1024*1024):.2f} MB)")
else:
    print(f"  ❌ Persistent Backup missing at: {drive_weights} (Double check if Drive is fully mounted)")

=== LOCATING YOUR TRAINED MODELS ===

📍 [Local Workspace Directory - /content/]
  ❌ Model Weights missing at: /content/mobilenetv3_crop_diseases.pth
  ❌ Full Model missing at:      /content/mobilenetv3_crop_diseases_full.pth

📍 [Google Drive Backup Directory]
  ❌ Persistent Backup missing at: /content/drive/MyDrive/crop_disease_model/mobilenetv3_crop_diseases.pth (Double check if Drive is fully mounted)


In [ ]:
import torch
import os
import shutil
from google.colab import drive

print("--- RE-SAVING TRAINED MODEL FROM MEMORY ---")

# Ensure the model variable exists in local memory
if 'model' in globals():
    # 1. Save local weights
    weights_filename = "/content/mobilenetv3_crop_diseases.pth"
    torch.save(model.state_dict(), weights_filename)
    print(f"✓ Successfully saved weights locally to: {weights_filename}")

    # 2. Save full local model
    full_model_filename = "/content/mobilenetv3_crop_diseases_full.pth"
    torch.save(model, full_model_filename)
    print(f"✓ Successfully saved full model locally to: {full_model_filename}")

    # 3. Save to Google Drive backup
    try:
        print("\nMounting Google Drive to backup your files...")
        drive.mount('/content/drive', force_remount=True)

        destination_dir = "/content/drive/MyDrive/crop_disease_model/"
        os.makedirs(destination_dir, exist_ok=True)

        drive_dest = os.path.join(destination_dir, "mobilenetv3_crop_diseases.pth")
        shutil.copy(weights_filename, drive_dest)
        print(f"✓ Successfully backed up weights to Google Drive: {drive_dest}")
    except Exception as e:
        print(f"⚠ Could not backup to Google Drive: {e}")
else:
    print("❌ Error: 'model' variable is no longer in memory. You may need to re-run the training cell (4cce9772) first.")

--- RE-SAVING TRAINED MODEL FROM MEMORY ---
✓ Successfully saved weights locally to: /content/mobilenetv3_crop_diseases.pth
✓ Successfully saved full model locally to: /content/mobilenetv3_crop_diseases_full.pth

Mounting Google Drive to backup your files...
Mounted at /content/drive
✓ Successfully backed up weights to Google Drive: /content/drive/MyDrive/crop_disease_model/mobilenetv3_crop_diseases.pth
